#1 Import functions 

In [ ]:
from sentence_transformers import SentenceTransformer, util
import torch
import pandas as pd
import numpy as np
from PIL import Image
import requests
import matplotlib.pyplot as plt
import chromadb

#3 Import the pre-trained CLIP model using SentenceTransformer

In [21]:
model = SentenceTransformer('clip-ViT-B-32')

#4 Establish connection to ChromaDB

In [23]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get the API key and other details from environment variables
api_key = os.getenv("CHROMA_API_KEY")
tenant_id = os.getenv("CHROMA_TENANT")
database_name = os.getenv("CHROMA_DATABASE")

# Check if the variables were loaded correctly
if not api_key:
    print("Error: API key not found. Make sure it's in your .env file.")
else:
    # Use the variables to create the client
    client = chromadb.CloudClient(
        api_key=api_key,
        tenant=tenant_id,
        database=database_name
    )
    print("ChromaDB client initialized successfully!")

ChromaDB client initialized successfully!


#5 Load the df

In [ ]:
file_path = "/kaggle/input/wikiart-all-artpieces/wikiart_art_pieces.csv"
df = pd.read_csv(file_path)

#6 Define an uploader for .jpg, .jpeg. and .png images 

In [ ]:
"""This was the code to embed the images into ChromaDA"""

# Select a small number of images to work with / batch it
small_df = df[1:1001]

# Create a list to store the image embeddings
image_embeddings = []

# Loop through the images, download them, and create their embeddings
for index, row in small_df.iterrows():
    try:
        image_url = row['img']
        image = Image.open(requests.get(image_url, stream=True).raw)

        # Use the CLIP model to encode the image.
        # The SentenceTransformer library's encode method automatically handles preprocessing.
        embedding = model.encode(image)

        # add the embedding to ChromaDB
        collection.add(
            embeddings=[embedding.tolist()],
            documents=[row['file_name']],
            metadatas=[{"artist": row['artist'], "style": row['style'], "url": image_url, "file_name": row['file_name']}],
            ids=[str(index)]
        )
        print(f"Processed and added image {index} to ChromaDB.")

    except Exception as e:
        print(f"Could not process image at URL {image_url}: {e}")